# Ligand-Receptor Fingerprint Analysis (D2 & D4)

Semester project: ML classification and **SHAP-based** interpretability of ProLIF fingerprints.

**Goals:**
1. Classify active vs inactive ligands
2. Identify key amino acids via **SHAP** (primary) and feature importance (comparison)
3. Compare SHAP with Fisher's exact test
4. Compare homologous positions between D2 and D4

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import dataset_summary, load_receptor_dataset
from src.models import train_and_evaluate, select_best_model
from src.interpretability import top_features_by_method, compare_method_overlap
from src.statistics import feature_statistical_tests, aggregate_stats_to_residues, compare_ml_vs_statistics
from src.homology import homology_overlap_summary, compare_homologous_residues, build_bw_homology_table

sns.set_theme(style='whitegrid')

## 1. Data overview

In [2]:
for receptor in ['d2', 'd4']:
    print(json.dumps(dataset_summary(receptor, '..'), indent=2))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our

{
  "receptor": "d2",
  "n_samples": 6112,
  "n_active": 3044,
  "n_inactive": 3068,
  "balance_ratio": 0.9921773142112125,
  "n_features": 110,
  "mean_interactions_per_sample": 16.364692408376964
}
{
  "receptor": "d4",
  "n_samples": 1514,
  "n_active": 1036,
  "n_inactive": 478,
  "balance_ratio": 2.1673640167364017,
  "n_features": 119,
  "mean_interactions_per_sample": 13.41347424042272
}


[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than ou

## 2. Train classifiers

For **D4**, class imbalance (~2.15:1 active:inactive) is handled by **random undersampling** of the majority (active) class in each training fold. This keeps fingerprint features binary (0/1), unlike SMOTE.

Set `use_undersampling=False` to use class weights instead.

In [3]:
results = {}
for receptor in ['d2', 'd4']:
    X, y = load_receptor_dataset(receptor, '..')
    use_undersampling = receptor == 'd4'
    model_results = train_and_evaluate(X, y, receptor, use_undersampling=use_undersampling)
    best = select_best_model(model_results)
    results[receptor] = {'X': X, 'y': y, 'best': best, 'all': model_results}
    print(f"\n{receptor.upper()} best: {best.name} (strategy: {best.imbalance_strategy})")
    for r in model_results:
        print(f"  {r.name}: balanced_accuracy={r.cv_metrics['balanced_accuracy']:.3f}, ROC-AUC={r.cv_metrics['roc_auc']:.3f}")

[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:37:47] Depickling from a version number (16.2)that is higher than ou


D2 best: random_forest (strategy: none)
  random_forest: balanced_accuracy=0.640, ROC-AUC=0.691
  logistic_l1: balanced_accuracy=0.596, ROC-AUC=0.639
  xgboost: balanced_accuracy=0.626, ROC-AUC=0.678


[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:38:01] Depickling from a version number (16.2)that is higher than ou


D4 best: random_forest (strategy: undersample)
  random_forest: balanced_accuracy=0.671, ROC-AUC=0.717
  logistic_l1: balanced_accuracy=0.642, ROC-AUC=0.677
  xgboost: balanced_accuracy=0.657, ROC-AUC=0.689


## 2.1. CV fold stability hypothesis

Are the same amino acids ranked highest in **every** cross-validation fold?

In [4]:
from src.cv_stability import cv_residue_stability, stability_summary

for receptor in ['d2', 'd4']:
    best = results[receptor]['best']
    X, y = results[receptor]['X'], results[receptor]['y']
    use_undersampling = receptor == 'd4'
    stability = cv_residue_stability(
        X, y, model_name=best.name, receptor=receptor,
        use_undersampling=use_undersampling, top_n=15,
    )
    summary = stability_summary(stability)
    print(f"\n{receptor.upper()} – residues in ALL folds: {summary['core_residues_in_all_folds']}")
    print(f"  Mean pairwise Jaccard: {summary['mean_pairwise_jaccard']:.3f}")
    display(stability.frequency.head(15))


D2 – residues in ALL folds: ['ASP114.A', 'HIS393.A', 'LEU94.A', 'PHE110.A', 'PHE189.A', 'PHE389.A', 'PHE390.A', 'TRP100.A', 'TRP386.A', 'TRP413.A', 'TYR408.A', 'TYR416.A', 'VAL115.A', 'VAL91.A']
  Mean pairwise Jaccard: 0.925


,residue,aa,position,folds_in_top_n,fold_fraction,stable
0,ASP114.A,ASP,114,5,1.0,True
1,HIS393.A,HIS,393,5,1.0,True
2,LEU94.A,LEU,94,5,1.0,True
3,PHE110.A,PHE,110,5,1.0,True
4,PHE189.A,PHE,189,5,1.0,True
5,PHE389.A,PHE,389,5,1.0,True
6,PHE390.A,PHE,390,5,1.0,True
7,TRP100.A,TRP,100,5,1.0,True
8,TRP386.A,TRP,386,5,1.0,True
9,TRP413.A,TRP,413,5,1.0,True



D4 – residues in ALL folds: ['ASP115.A', 'CYS185.A', 'HIS414.A', 'LEU111.A', 'LEU187.A', 'MET112.A', 'PHE410.A', 'PHE411.A', 'PHE91.A', 'SER94.A', 'TRP101.A', 'TYR438.A', 'VAL116.A', 'VAL193.A']
  Mean pairwise Jaccard: 0.912


,residue,aa,position,folds_in_top_n,fold_fraction,stable
0,ASP115.A,ASP,115,5,1.0,True
1,CYS185.A,CYS,185,5,1.0,True
2,HIS414.A,HIS,414,5,1.0,True
3,LEU111.A,LEU,111,5,1.0,True
4,LEU187.A,LEU,187,5,1.0,True
5,MET112.A,MET,112,5,1.0,True
6,PHE410.A,PHE,410,5,1.0,True
7,PHE411.A,PHE,411,5,1.0,True
8,PHE91.A,PHE,91,5,1.0,True
9,SER94.A,SER,94,5,1.0,True


## 3. Interpretability analysis

In [5]:
interp_results = {}
for receptor in ['d2', 'd4']:
    best = results[receptor]['best']
    X, y = results[receptor]['X'], results[receptor]['y']
    interp = top_features_by_method(best.model, X, y, top_n=20)
    overlap = compare_method_overlap(interp, top_n=15)
    interp_results[receptor] = interp
    print(f"\n{receptor.upper()} – method overlap:")
    display(overlap)


D2 – method overlap:


,method_1,method_2,overlap,jaccard
0,native_residues,shap_residues,13,0.764706



D4 – method overlap:


,method_1,method_2,overlap,jaccard
0,native_residues,shap_residues,11,0.647059


## 4. Statistical analysis (active vs inactive)

In [6]:
stat_results = {}
for receptor in ['d2', 'd4']:
    X, y = results[receptor]['X'], results[receptor]['y']
    stats = feature_statistical_tests(X, y)
    stat_residues = aggregate_stats_to_residues(stats, top_n=20)
    stat_results[receptor] = {'stats': stats, 'residues': stat_residues}
    
    ml_key = 'shap_residues' if 'shap_residues' in interp_results[receptor] else 'permutation_residues'
    comparison = compare_ml_vs_statistics(interp_results[receptor][ml_key], stat_residues)
    print(f"\n{receptor.upper()} ML vs Statistics (top 15):")
    print(f"  Overlap: {comparison['n_overlap']} residues, Jaccard={comparison['jaccard']:.2f}")
    print(f"  Shared: {comparison['overlap']}")


D2 ML vs Statistics (top 15):
  Overlap: 12 residues, Jaccard=0.67
  Shared: ['HIS393.A', 'ILE184.A', 'LEU94.A', 'PHE189.A', 'PHE198.A', 'PHE389.A', 'SER409.A', 'THR412.A', 'TRP100.A', 'TRP413.A', 'TYR408.A', 'VAL91.A']

D4 ML vs Statistics (top 15):
  Overlap: 11 residues, Jaccard=0.65
  Shared: ['ARG186.A', 'ASP115.A', 'HIS414.A', 'LEU187.A', 'PHE410.A', 'PHE411.A', 'PHE91.A', 'THR434.A', 'TYR438.A', 'VAL116.A', 'VAL430.A']


## 5. Homology comparison (D2 vs D4)

Residues are mapped to **Ballesteros-Weinstein** numbering to compare homologous positions.

In [7]:
display(build_bw_homology_table())

ml_key = 'shap_residues' if 'shap_residues' in interp_results['d2'] else 'permutation_residues'
homology_ml = homology_overlap_summary(
    interp_results['d2'][ml_key],
    interp_results['d4'][ml_key],
    top_n=15
)
print('ML homology overlap:', homology_ml['n_shared'], 'shared BW positions')
print('Shared positions:', homology_ml['shared_bw_positions'])

comp = compare_homologous_residues(
    interp_results['d2'][ml_key],
    interp_results['d4'][ml_key],
    top_n=20
)
display(comp[comp['both_significant']])

,bw_id,d2_position,d4_position
0,3.28,100,101
1,3.29,103,103
2,3.32,114,115
3,4.56,178,178
4,4.59,181,181
5,4.60,184,111
6,4.66,190,195
7,4.68,192,192
8,5.43,389,91
9,5.58,404,404


ML homology overlap: 3 shared BW positions
Shared positions: ['3.28', '3.32', '5.43']


,bw_id,d2_position,d4_position,d2_residue,d2_aa,d2_position_data,d2_score,d4_residue,d4_aa,d4_position_data,d4_score,both_significant
0,3.28,100,101,TRP100.A,TRP,100.0,0.020218,TRP101.A,TRP,101.0,0.010762,True
2,3.32,114,115,ASP114.A,ASP,114.0,0.009166,ASP115.A,ASP,115.0,0.019501,True
8,5.43,389,91,PHE389.A,PHE,389.0,0.022662,PHE91.A,PHE,91.0,0.025994,True


## 6. Run full pipeline (optional)

Saves all CSVs and plots to `results/`.

In [8]:
from src.run_analysis import run_full_analysis
run_full_analysis(data_dir='..', output_dir='../results')


Processing receptor D2


[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than ou

{
  "receptor": "d2",
  "n_samples": 6112,
  "n_active": 3044,
  "n_inactive": 3068,
  "balance_ratio": 0.9921773142112125,
  "n_features": 110,
  "mean_interactions_per_sample": 16.364692408376964
}


[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:43:14] Depickling from a version number (16.2)that is higher than ou


Best model: random_forest (balanced_accuracy=0.640)
  Imbalance strategy: none
  random_forest: {'accuracy': 0.6397256350927998, 'balanced_accuracy': 0.6397168428273885, 'f1': 0.6380429316526872, 'roc_auc': 0.6906804776323942}
  logistic_l1: {'accuracy': 0.5965329011726952, 'balanced_accuracy': 0.5964678546032809, 'f1': 0.5885724051844455, 'roc_auc': 0.6391971948162096}
  xgboost: {'accuracy': 0.6254912325544361, 'balanced_accuracy': 0.6255023295435681, 'f1': 0.6251036457421089, 'roc_auc': 0.6783085197090871}

  CV stability (top-15 residues across folds):
    Core residues (all folds): ['ASP114.A', 'HIS393.A', 'LEU94.A', 'PHE110.A', 'PHE189.A', 'PHE389.A', 'PHE390.A', 'TRP100.A', 'TRP386.A', 'TRP413.A', 'TYR408.A', 'TYR416.A', 'VAL115.A', 'VAL91.A']
    Mean pairwise Jaccard: 0.925
    Hypothesis supported: True


...

[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:48:52] Depickling from a version number (16.2)that is higher than ou


Best model: random_forest (balanced_accuracy=0.671)
  Imbalance strategy: undersample
  random_forest: {'accuracy': 0.6902235918956134, 'balanced_accuracy': 0.6705500425394589, 'f1': 0.7614387102287516, 'roc_auc': 0.7172267084237907}
  logistic_l1: {'accuracy': 0.6565449260157804, 'balanced_accuracy': 0.6419909933697119, 'f1': 0.7306185073987872, 'roc_auc': 0.6765536885459654}
  xgboost: {'accuracy': 0.6671125390684763, 'balanced_accuracy': 0.6565047600187761, 'f1': 0.738103598808989, 'roc_auc': 0.6890423895114971}

  CV stability (top-15 residues across folds):
    Core residues (all folds): ['ASP115.A', 'CYS185.A', 'HIS414.A', 'LEU111.A', 'LEU187.A', 'MET112.A', 'PHE410.A', 'PHE411.A', 'PHE91.A', 'SER94.A', 'TRP101.A', 'TYR438.A', 'VAL116.A', 'VAL193.A']
    Mean pairwise Jaccard: 0.912
    Hypothesis supported: True


[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than our version (15.0).
This probably won't work.
[19:49:39] Depickling from a version number (16.2)that is higher than ou

Figures saved to presentation/figures/

Results saved to ../results/


{'summaries': {'d2': {'receptor': 'd2',
   'n_samples': 6112,
   'n_active': 3044,
   'n_inactive': 3068,
   'balance_ratio': 0.9921773142112125,
   'n_features': 110,
   'mean_interactions_per_sample': 16.364692408376964},
  'd4': {'receptor': 'd4',
   'n_samples': 1514,
   'n_active': 1036,
   'n_inactive': 478,
   'balance_ratio': 2.1673640167364017,
   'n_features': 119,
   'mean_interactions_per_sample': 13.41347424042272}},
 'receptors': {'d2': {'best_model': 'random_forest',
   'cv_metrics': {'random_forest': {'accuracy': 0.6397256350927998,
     'balanced_accuracy': 0.6397168428273885,
     'f1': 0.6380429316526872,
     'roc_auc': 0.6906804776323942},
    'logistic_l1': {'accuracy': 0.5965329011726952,
     'balanced_accuracy': 0.5964678546032809,
     'f1': 0.5885724051844455,
     'roc_auc': 0.6391971948162096},
    'xgboost': {'accuracy': 0.6254912325544361,
     'balanced_accuracy': 0.6255023295435681,
     'f1': 0.6251036457421089,
     'roc_auc': 0.6783085197090871}},
  